<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/03_dataset_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Dataset report

What is in the merged dataset, is it sane, and how should training balance it?

```
reads   MyDrive/clear_vision/datasets/merged.zip + merged.done   (prepare_dataset, unpacked to /content)
        MyDrive/clear_vision/datasets/merge_report.json, pseudo_labels.csv (if notebook 09 ran)
writes  MyDrive/clear_vision/datasets/train_balanced.txt   train image list with rare classes repeated
        MyDrive/clear_vision/datasets/data_balanced.yaml   data.yaml whose train is that list
```

- **Images are never read from Drive.** Only these two small text files are written back.
- **Charts:**
  - images per split
  - boxes per class per split (original vs pseudo-labeled, if notebook 09 ran)
  - box sizes per class
  - boxes per source per class
- **Grid per class:** a dropdown shows 16 random images of any class, to catch bad
  mappings by eye.
- **Class table:** classes under 300 train boxes are marked "needs more data".
- **Sanity checks:**
  - no image in two splits (by name and by content)
  - class IDs within 0-19
  - no empty label files in val/test
  - image counts match `merge_report.json`
- **Balancing:**
  - Images of rare focus classes are repeated, at most 4x.
  - Person-only images are capped at 4,000.
  - Training applies the result after `prepare_dataset()` with
    `src/colab_data.py apply_balanced()`, which refuses a list made for an older zip.
- **Runtime:** CPU is fine. Run all cells. Re-run after notebook 02 or 09 changes the zip.

In [ ]:
# Settings
REPO_URL = "https://github.com/Zikirullo/clear-vision.git"
BRANCH = "develop"
DRIVE_ROOT = "/content/drive/MyDrive/clear_vision"
MERGED_ZIP = f"{DRIVE_ROOT}/datasets/merged.zip"
MIN_TRAIN_BOXES = 300  # fewer train boxes than this -> "needs more data"
HASH_IMAGES = True  # also compare image contents (md5) across splits, not only names
# Balancing
FOCUS_CLASSES = ["curb", "pothole", "barrier", "crosswalk", "manhole", "pole", "bollard",
                 "scooter", "fire_hydrant", "bus"]
MAX_REPEAT = 4  # an image is listed at most this many times
TARGET_IMAGES = None  # repeat = ceil(TARGET / train images with the class); None = median over classes
PERSON_CAP = 4000  # keep at most this many person-only train images
SEED = 42

## 1. Setup

In [ ]:
# 1. Setup: Drive, repo, packages
import json
import logging
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = Path("/content/clear-vision")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print(f"Repo at {REPO_DIR}: {commit.strip()}")

%pip install -q ultralytics

sys.path.insert(0, str(REPO_DIR))
import scripts.dataset_report as dr  # noqa: E402
from src.colab_data import prepare_dataset  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
MERGED_ZIP = Path(MERGED_ZIP)

## 2. Load the dataset

Unpacks the verified zip to `/content/datasets/merged/`. Then it reads every label file
and every image header (for box sizes in pixels at the 640 px training size).

In [ ]:
# 2. Unpack and scan
DATA_YAML = prepare_dataset(MERGED_ZIP)
DATASET = DATA_YAML.parent
REPORT = dr.load_report(MERGED_ZIP.with_name("merge_report.json"))
CLASSES = list(REPORT["classes"])
PSEUDO_CSV = MERGED_ZIP.with_name("pseudo_labels.csv")
HAS_PSEUDO = PSEUDO_CSV.exists() and "pseudo_labels" in REPORT

SCAN = dr.scan_dataset(DATASET, CLASSES, list(REPORT["sources"]), PSEUDO_CSV if HAS_PSEUDO else None)
print(f"{sum(dr.images_per_split(SCAN).values())} images, {len(SCAN.boxes)} boxes, {len(CLASSES)} classes, "
      f"{len(REPORT['sources'])} sources, pseudo-labels: {'yes' if HAS_PSEUDO else 'no'}")

## 3. Charts

In [ ]:
# 3.0 Chart style: one blue for single series, fixed categorical order, thin bars with gaps
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#9a9a94"
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]  # + "Other" gray
SIZE_COLORS = {"small": "#86b6ef", "medium": "#3987e5", "large": "#184f95"}  # ordinal: light -> dark
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#b5b4ad",
    "axes.grid": True, "axes.grid.axis": "x", "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "text.color": "#2b2b28", "axes.labelcolor": "#56554f", "xtick.color": "#56554f", "ytick.color": "#2b2b28",
})
BAR = {"height": 0.7, "edgecolor": "white", "linewidth": 1.5}  # 2px-ish surface gap between segments

In [ ]:
# 3.1 Images per split
counts = dr.images_per_split(SCAN)
fig, ax = plt.subplots(figsize=(7, 2.2))
bars = ax.barh(list(counts)[::-1], list(counts.values())[::-1], color=BLUE, **BAR)
ax.bar_label(bars, labels=[f"{v:,}" for v in list(counts.values())[::-1]], padding=4)
ax.set_title("Images per split")
ax.set_xlabel("images")
ax.margins(x=0.12)
plt.show()

In [ ]:
# 3.2 Boxes per class per split: original vs pseudo-labeled (stacked)
per = dr.boxes_per_class(SCAN)
fig, axes = plt.subplots(1, 3, figsize=(16, 7), sharey=True)
y = np.arange(len(CLASSES))[::-1]
for ax, split in zip(axes, dr.SPLITS):
    orig = np.array([per[split][c]["original"] for c in CLASSES])
    pseudo = np.array([per[split][c]["pseudo"] for c in CLASSES])
    ax.barh(y, orig, color=BLUE, label="original", **BAR)
    if HAS_PSEUDO:
        ax.barh(y, pseudo, left=orig, color=ORANGE, label="pseudo-labeled", **BAR)
    for yi, total in zip(y, orig + pseudo):
        ax.text(total, yi, f" {total:,}", va="center", fontsize=8, color="#56554f")
    ax.set_title(f"{split}: {int((orig + pseudo).sum()):,} boxes")
    ax.set_xlabel("boxes")
    ax.margins(x=0.18)
axes[0].set_yticks(y, CLASSES)
if HAS_PSEUDO:
    axes[0].legend(loc="lower right", frameon=False)
fig.suptitle("Boxes per class per split" + (" (orange = added by notebook 09)" if HAS_PSEUDO else ""), x=0.01, ha="left")
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Box size per class: share of small / medium / large boxes at the 640 px training size
sizes = dr.size_distribution(SCAN)
fig, ax = plt.subplots(figsize=(10, 7))
left = np.zeros(len(CLASSES))
for size in dr.SIZES:
    n = np.array([sizes[c][size] for c in CLASSES], dtype=float)
    share = np.divide(n, [max(1, sum(sizes[c].values())) for c in CLASSES])
    ax.barh(y, share, left=left, color=SIZE_COLORS[size], label=size, **BAR)
    left += share
for yi, c in zip(y, CLASSES):
    ax.text(1.01, yi, f"{sum(sizes[c].values()):,}", va="center", fontsize=8, color="#56554f")
ax.set_yticks(y, CLASSES)
ax.set_xlim(0, 1.08)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}" if v <= 1 else ""))
ax.set_title("Box size per class (small < 32², medium < 96², large; px² at 640 px)  ·  right: total boxes")
ax.legend(ncols=3, loc="upper center", bbox_to_anchor=(0.5, -0.06), frameon=False)
plt.show()
display(pd.DataFrame(sizes).T.rename_axis("class"))

In [ ]:
# 3.4 Boxes per source per class (stacked; the 7 biggest sources, the rest as "Other")
by_source = dr.boxes_per_source(SCAN)
totals = pd.DataFrame(by_source).fillna(0).astype(int).T  # class x source
order = totals.sum().sort_values(ascending=False).index.tolist()
top, rest = order[:7], order[7:]
fig, ax = plt.subplots(figsize=(12, 7))
left = np.zeros(len(CLASSES))
for i, src in enumerate(top + (["Other"] if rest else [])):
    n = (totals[rest].sum(axis=1) if src == "Other" else totals[src]).reindex(CLASSES).to_numpy()
    ax.barh(y, n, left=left, color=GRAY if src == "Other" else CATEGORICAL[i], label=src, **BAR)
    left += n
ax.set_yticks(y, CLASSES)
ax.set_xlabel("boxes (all splits)")
ax.set_title("Boxes per source per class" + (f'  ·  Other = {", ".join(rest)}' if rest else ""))
ax.legend(loc="lower right", frameon=False, title="source")
plt.show()
display(totals.reindex(CLASSES)[order].rename_axis("class"))  # exact numbers, all sources

## 4. Look at a class

Pick a class to see 16 random images that have it:
- **yellow:** original boxes of that class
- **orange:** its pseudo-labels
- **thin gray:** other classes

A wrong mapping (for example, kerbstones labeled as stairs) shows up here. "Shuffle"
draws a new sample.

In [ ]:
# 4. Class grid with a dropdown
import cv2
import ipywidgets as widgets
from IPython.display import clear_output

choice = widgets.Dropdown(options=CLASSES, value="stairs" if "stairs" in CLASSES else CLASSES[0], description="class")
shuffle = widgets.Button(description="Shuffle", icon="random")
out = widgets.Output()


def show(*_):
    with out:
        clear_output(wait=True)
        grid = dr.example_grid(DATASET, SCAN, choice.value)
        if grid is None:
            print(f"No image has a {choice.value} box.")
            return
        n = len(dr.images_with_class(SCAN, choice.value))
        plt.figure(figsize=(18, 13.5))
        plt.imshow(cv2.cvtColor(grid, cv2.COLOR_BGR2RGB))
        plt.title(f"{choice.value}: 16 of {n:,} images", loc="left")
        plt.axis("off")
        plt.show()


choice.observe(show, names="value")
shuffle.on_click(show)
display(widgets.HBox([choice, shuffle]), out)
show()

## 5. Classes and data needs

In [ ]:
# 5. Class table; classes under MIN_TRAIN_BOXES train boxes need more data
ROWS = dr.class_table(SCAN, MIN_TRAIN_BOXES)
print(dr.format_rows(ROWS))
needs = [r for r in ROWS if r["status"] != "ok"]
print(f"\nNeeds more data (< {MIN_TRAIN_BOXES} train boxes): {', '.join(r['class'] for r in needs) or 'none'}")
display(pd.DataFrame(ROWS).set_index("class").style.apply(
    lambda row: ["background-color: #fde2d6; color: #000" if row["status"] != "ok" else "" for _ in row], axis=1))

## 6. Sanity checks

- **FAIL:** the dataset is wrong. Balancing (section 7) refuses to write until it is
  fixed, usually by re-running notebook 02.
- **WARN:** worth a look. Empty label files in val/test are background images (images
  without objects, kept on purpose by the merge), not broken files.

In [ ]:
# 6. Sanity checks
CHECKS = dr.sanity_checks(DATASET, SCAN, REPORT, hash_images=HASH_IMAGES)
for c in CHECKS:
    print(f"[{c.status}] {c.name}: {c.detail}")
FAILED = [c for c in CHECKS if c.status == "FAIL"]
print("\nAll hard checks passed." if not FAILED else f"\n{len(FAILED)} check(s) FAILED: fix the dataset before training.")

## 7. Balanced train list

- **Repeats:** an image with a focus class is listed `ceil(TARGET / train images with
  that class)` times, between 1 and `MAX_REPEAT`. If it has several focus classes, the
  largest repeat wins. `TARGET` defaults to the median train images per class.
- **Person cap:** person-only images (every box is a person) are randomly capped at
  `PERSON_CAP`.
- **Everything else** is listed once, background images included.
- **Lines** look like `./images/train/<name>`, relative to the dataset root. Ultralytics
  reads a `./` line relative to the list file's folder, and `apply_balanced()` puts the
  list in the dataset root.

In [ ]:
# 7. Write train_balanced.txt + data_balanced.yaml next to merged.zip
if FAILED:
    raise RuntimeError("Sanity checks failed (section 6): not writing a balanced list for a broken dataset.")

LINES, SUMMARY = dr.balanced_train(SCAN, FOCUS_CLASSES, MAX_REPEAT, PERSON_CAP, TARGET_IMAGES, SEED)
print(f"target images per class: {SUMMARY['target_images']}   repeat factors: {SUMMARY['factors']}")
print(f"train images {SUMMARY['train_images']:,} -> {SUMMARY['lines']:,} lines; "
      f"{SUMMARY['repeated_images']:,} images repeated; "
      f"{SUMMARY['person_only_dropped']:,} of {SUMMARY['person_only_images']:,} person-only images dropped")

balance = pd.DataFrame({"train boxes": SUMMARY["boxes_before"], "balanced": SUMMARY["boxes_after"]})
balance["x"] = (balance["balanced"] / balance["train boxes"].clip(lower=1)).round(2)
balance["focus"] = ["yes" if c in FOCUS_CLASSES else "" for c in balance.index]
display(balance.rename_axis("class"))

done = json.loads(MERGED_ZIP.with_suffix(".done").read_text())
TXT, YML = dr.write_balanced(MERGED_ZIP.parent, LINES, CLASSES, done, SUMMARY)
for p in (TXT, YML):
    print(f"wrote {p} ({p.stat().st_size / 1e6:.2f} MB)")

## Using it in training

```python
from src.colab_data import prepare_dataset, apply_balanced

prepare_dataset(MERGED_ZIP)               # unpack + check, as always
data_yaml = apply_balanced(MERGED_ZIP)    # copies the list into the dataset, returns data_balanced.yaml
model.train(data=str(data_yaml), ...)     # val/test stay the normal, unbalanced splits
```

`apply_balanced()` fails if the list was made for a different `merged.zip` (after
notebook 02 or 09 changed it). Re-run this notebook then.